In [1]:
from src import *
import torch

In [2]:
#d = 2
#
#def u_fun(x):
#    # takes in a matrix, return a vector
#    return torch.prod(torch.sin(torch.pi * x), dim=1)
#def f_fun(x):
#    # takes in a matrix, return a vector
#    return -d * torch.pi**2 * torch.prod(torch.sin(torch.pi * x), dim=1)

# Example usage for Poisson equation: laplace u = f
#model = PINN(d)
#x = torch.randn(100, d, requires_grad=True)
#u, grad_u, laplace_u = compute_derivatives(model, x)
#f = f_fun(x)
#pde_loss = ((laplace_u.squeeze() - f)**2).mean()

d=1

def f(x):
    return x**2

def f_der(x):
    return 2*x

def f_der_der(x):
    return 2

In [3]:
class SIMPLE(nn.Module):
    def __init__(self, d, hidden_size=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d, hidden_size),
            nn.Linear(hidden_size, 1),
        )
    
    def forward(self, x):
        return self.net(x)

In [4]:
d = 1
#x = torch.tensor([[-0.8], [-0.4], [0.0], [0.5], [1.0]], requires_grad=True)
#x = torch.tensor([[-0.4]], requires_grad=True)
x = torch.randn(3, d, requires_grad=True)
x

tensor([[-0.0320],
        [ 1.4705],
        [ 1.3469]], requires_grad=True)

In [5]:
lr = 1e-3
model = SIMPLE(d)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
optimizer.zero_grad()

In [6]:
model.parameters

<bound method Module.parameters of SIMPLE(
  (net): Sequential(
    (0): Linear(in_features=1, out_features=64, bias=True)
    (1): Linear(in_features=64, out_features=1, bias=True)
  )
)>

In [7]:
model.net[0].weight

Parameter containing:
tensor([[ 0.5682],
        [-0.7491],
        [ 0.3097],
        [-0.4590],
        [ 0.5163],
        [-0.7037],
        [ 0.5131],
        [-0.8675],
        [-0.6828],
        [-0.0167],
        [-0.4078],
        [-0.1828],
        [-0.4967],
        [ 0.2249],
        [ 0.5347],
        [-0.7233],
        [-0.0833],
        [-0.9534],
        [-0.8917],
        [ 0.2393],
        [ 0.8630],
        [ 0.0331],
        [-0.1984],
        [-0.3373],
        [ 0.6722],
        [ 0.8670],
        [-0.5891],
        [ 0.0965],
        [ 0.5534],
        [ 0.6184],
        [-0.0575],
        [-0.7137],
        [-0.1969],
        [ 0.8658],
        [-0.2961],
        [-0.2198],
        [ 0.1990],
        [ 0.1264],
        [ 0.6353],
        [ 0.0707],
        [-0.5443],
        [ 0.5675],
        [-0.1667],
        [ 0.6455],
        [-0.0696],
        [ 0.4467],
        [-0.1529],
        [ 0.7692],
        [-0.4355],
        [ 0.8139],
        [-0.7141],
        [

In [8]:
u = model(x)
u

tensor([[-0.7150],
        [-1.0008],
        [-0.9773]], grad_fn=<AddmmBackward0>)

In [9]:
g1 = torch.autograd.grad(
    outputs=u,
    inputs=x,
    grad_outputs=torch.ones_like(u),
    create_graph=True,
)
grad_u = g1[0]
grad_u

tensor([[-0.1903],
        [-0.1903],
        [-0.1903]], grad_fn=<MmBackward0>)

In [10]:
grad_u[:, 0].sum()

tensor(-0.5708, grad_fn=<SumBackward0>)

In [11]:
g2 = torch.autograd.grad(
    outputs=grad_u[:,0].sum(),
    inputs=x,
    create_graph=True,
    allow_unused=True
)
g2

(None,)

---

In [12]:
lr = 1e-3
model = SIMPLE(d)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
optimizer.zero_grad()

---

In [13]:
x = torch.tensor([2.06, 5.7], requires_grad=True)
t = torch.tensor([1.5, 0.33], requires_grad=True)

model = torch.nn.Linear(2, 1)

var_input = torch.stack([x, t], dim=1)

u = model(var_input)

du_dt = torch.autograd.grad(u.sum(), t, create_graph=True)[0]
du_dx = torch.autograd.grad(u.sum(), x, create_graph=True)[0]
d2u_dx2 = torch.autograd.grad(du_dx.sum(), x, allow_unused=True)[0]

print(u)
print(du_dt)
print(du_dx)
print(d2u_dx2)

tensor([[-0.9443],
        [-0.7568]], grad_fn=<AddmmBackward0>)
tensor([-0.2163, -0.2163], grad_fn=<SelectBackward0>)
tensor([-0.0180, -0.0180], grad_fn=<SelectBackward0>)
None
